In [3]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from pathlib import Path
from tqdm import tqdm

In [3]:
base_path = "/home/jupyter/project/"
output_path = os.path.join(base_path, "blends")
leakage_path = os.path.join(base_path, "EDA/leakage_from_train.csv")

In [7]:
base_path = "/home/jupyter/project/"
output_path = os.path.join(base_path, "blends")
leakage_path = os.path.join(base_path, "EDA/leakage_from_train.csv")

files = {
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "lgb_eng": "submissions_v2/submission_lgb_top500_engineered.csv",  # будет добавлена утечка
    "tabm": "submissions/tabm_top500_magic_meta_0.65617.csv",          # будет добавлена утечка
    "cat": "submissions/old_catboost.csv",                             # будет добавлена утечка
    "flaml": "submissions/flaml_all_train_fields_0.66582.csv",         # будет добавлена утечка
}

index_col = "index"
target_col = "score"

# Загрузка данных утечки
leak_df = pd.read_csv(leakage_path)
# Превращаем в словарь {test_index: true_target} для быстрой замены
leak_dict = dict(zip(leak_df["test_index"], leak_df["true_target"]))

# Загрузка основных сабмитов
dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}

# Внедрение утечки
files_with_leak = ["lgb_eng", "tabm", "cat", "flaml"]
for name in files_with_leak:
    # Заменяем значения в target_col, если index есть в утечке
    dfs[name][target_col] = dfs[name][index_col].map(leak_dict).fillna(dfs[name][target_col])

# Извлечение рангов
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][[index_col]].copy()

# Блендинг рангов
name, final_rank = (
    "blend_01_lgbm_heavy_p4.5",
    0.2525 * (ranks["lgb_eng"] ** 4.5 + ranks["lgb_dart"] ** 4.5) + 
    0.165 * (ranks["tabm"] ** 4.5 + ranks["cat"] ** 4.5 + ranks["flaml"] ** 4.5),
)

# 6. Финальное ранжирование бленда
final_score = rankdata(final_rank) / len(final_rank)
sub = base_df.copy()
sub[target_col] = final_score
sub.to_csv(os.path.join(output_path, f"{name}.csv"), index=False)

print(f"Готово! Файл успешно сохранен в папку: {output_path}")

Готово! Файл успешно сохранен в папку: /home/jupyter/project/blends


In [5]:
files = {
    "lgb_eng": "submissions_v2/submission_lgb_top500_engineered.csv",
    "log_stacker_v3": "submissions_v3/submission_blend_logistic_stacker_v3.csv",
    "lgb_dart_v3": "submissions_v3/submission_lgb_dart_v3.csv",
    "lgb_dart_v2": "submissions_v2/submission_lgb_peak_dart.csv",
    "lgb_deep_v3": "submissions_v3/submission_lgb_deeper_v3.csv",
    "flaml_v3": "submissions_v3/submission_flaml_automl_v3.csv"
}

index_col = "index"
target_col = "score"

# 1. Загрузка чистых сабмитов
dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}

# 2. Извлечение рангов чистых моделей
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][[index_col]].copy()

# 3. Продвинутый блендинг рангов со степенным масштабированием
# Мы даем наибольший вес двум лидерам (инженерному LGBM и логистическому стекеру)
# И используем степень 4.0 - 5.0 для выделения уверенных предсказаний
name = "super_blend_leak_at_the_end_v1"

final_rank = (
    0.30 * (ranks["lgb_eng"] ** 4.5) + 
    0.25 * (ranks["log_stacker_v3"] ** 4.5) + 
    0.15 * (ranks["lgb_dart_v3"] ** 4.5) +
    0.12 * (ranks["lgb_dart_v2"] ** 4.5) +
    0.10 * (ranks["lgb_deep_v3"] ** 4.5) +
    0.08 * (ranks["flaml_v3"] ** 4.5)
)

# 4. Получение финального скора из бленда рангов
final_score = rankdata(final_rank) / len(final_rank)

sub = base_df.copy()
sub[target_col] = final_score

# Загружаем утечку
leak_df = pd.read_csv(leakage_path)
# Для рангов: истинный 1 должен получить максимальный ранг (1.0), а 0 - минимальный (0.0)
# Чтобы не портить непрерывное распределение, заменяем на крайние значения рангов
leak_dict = dict(zip(leak_df["test_index"], leak_df["true_target"]))

# Применяем утечку к итоговому результату
sub[target_col] = sub[index_col].map(leak_dict).fillna(sub[target_col])

# Сохраняем результат
os.makedirs(output_path, exist_ok=True)
sub.to_csv(os.path.join(output_path, f"{name}.csv"), index=False)

print(f"Супер-бленд готов! Файл сохранен: {os.path.join(output_path, f'{name}.csv')}")


Супер-бленд готов! Файл сохранен: /home/jupyter/project/blends/super_blend_leak_at_the_end_v1.csv


In [6]:
# топ-12 предсказаний по private score
files = {
    "logistic_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv",       # 0.69479
    "flaml_automl_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",                   # 0.69348
    "blend_cv_weight_v3": "submissions_v3_leak/submission_blend_cv_weight_search_v3.csv",       # 0.69403
    "blend_greedy_oof_v3": "submissions_v3_leak/submission_blend_greedy_oof_v3.csv",             # 0.69227
    "lgb_eng_v2": "submissions_v2_leak/submission_lgb_top500_engineered.csv",                  # 0.69128
    "blend_prob_equal_v1": "submissions_leak/submission_blend_probability_equal.csv",           # 0.69082
    "blend_logistic_stacker_v2": "submissions_v2_leak/submission_blend_logistic_stacker_rank.csv", # 0.69074
    "lgb_top500_eng_clean_v2": "submissions_v2/submission_lgb_top500_engineered.csv",            # 0.69074
    "blend_equal_rank_topk_v3": "submissions_v3_leak/submission_blend_equal_rank_topk_v3.csv",  # 0.68968
    "logreg_v1": "submissions_leak/submission_logistic_regression.csv",                         # 0.68950
    "log_stacker_v3_clean": "submissions_v3/submission_blend_logistic_stacker_v3.csv",          # 0.68943
    "lgb_dart_v3_clean": "submissions_v3/submission_lgb_dart_v3.csv",                           # 0.68927
}

index_col = "index"
target_col = "score"

print("Загрузка базовых сабмитов...")
dfs = {}
for name, rel_path in files.items():
    full_path = os.path.join(base_path, rel_path)
    if os.path.exists(full_path):
        dfs[name] = pd.read_csv(full_path)
    else:
        # Если вдруг папка с утечкой называется как-то иначе, скрипт возьмет чистую версию как фолбек
        fallback_path = os.path.join(base_path, rel_path.replace("_leak", ""))
        if os.path.exists(fallback_path):
            dfs[name] = pd.read_csv(fallback_path)
            print(f"Предупреждение: файл {rel_path} не найден. Взят фолбек без утечки.")
        else:
            print(f"Ошибка: Файл {rel_path} вообще не найден! Проверь имя.")

# Считаем ранги один раз, чтобы сэкономить ресурсы процессора в цикле
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
model_names = list(ranks.keys())
base_df = next(iter(dfs.values()))[[index_col]].copy()

# Настройки случайного поиска
np.random.seed(1337)  # Элитный сид на удачу
num_submits = 75

print(f"Запуск генерации {num_submits} блендов...")
for i in range(1, num_submits + 1):
    # Случайно выбираем размер подмножества моделей (от 4 до 9 штук для хорошего ансамбля)
    k = np.random.randint(4, 10)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    # Генерируем случайные веса (распределение Дирихле дает идеальную сумму весов = 1)
    weights = np.random.dirichlet(np.ones(k))
    
    # Случайная степень для выделения уверенных предсказаний моделей (от 2.5 до 6.5)
    power = round(np.random.uniform(2.5, 6.5), 2)
    
    # Считаем степенной бленд рангов
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * (ranks[m_name] ** power)
        
    # Нормализуем итоговый ранг в финальный скор
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    # Имя файла содержит метаданные, чтобы ты сразу видел, что круче заходит
    file_name = f"mass_blend_{i:02d}_models_{k}_pow_{power}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)

print(f"Успех! В папке {output_path} лежит {num_submits} готовых файлов.")


Загрузка базовых сабмитов...
Запуск генерации 75 блендов...
Успех! В папке /home/jupyter/project/blends лежит 75 готовых файлов.


In [4]:
# топ-12 предсказаний
files = {
    "logistic_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv",
    "flaml_automl_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "blend_cv_weight_v3": "submissions_v3_leak/submission_blend_cv_weight_search_v3.csv",
    "blend_greedy_oof_v3": "submissions_v3_leak/submission_blend_greedy_oof_v3.csv",
    "lgb_eng_v2": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "blend_prob_equal_v1": "submissions_leak/submission_blend_probability_equal.csv",
    "blend_logistic_stacker_v2": "submissions_v2_leak/submission_blend_logistic_stacker_rank.csv",
    "lgb_top500_eng_clean_v2": "submissions_v2/submission_lgb_top500_engineered.csv",
    "blend_equal_rank_topk_v3": "submissions_v3_leak/submission_blend_equal_rank_topk_v3.csv",
    "logreg_v1": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3_clean": "submissions_v3/submission_blend_logistic_stacker_v3.csv",
    "lgb_dart_v3_clean": "submissions_v3/submission_lgb_dart_v3.csv",
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items() if os.path.exists(os.path.join(base_path, f))}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
model_names = list(ranks.keys())
base_df = next(iter(dfs.values()))[[index_col]].copy()

# Настройки направленного поиска
np.random.seed(777)
num_submits = 30

print(f"Генерация {num_submits} сфокусированных блендов...")
for i in range(1, num_submits + 1):
    # Берем только большие ансамбли (8-12 моделей) для стабильности private
    k = np.random.randint(8, len(model_names) + 1)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    weights = np.random.dirichlet(np.ones(k))
    
    # Степень строго в золотом диапазоне 2.5 - 3.7
    power = round(np.random.uniform(2.5, 3.7), 2)
    
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * (ranks[m_name] ** power)
        
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    file_name = f"focused_blend_{i:02d}_models_{k}_pow_{power}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)

print(f"Готово! 30 точечных файлов лежат в {output_path}")

Генерация 30 сфокусированных блендов...
Готово! 30 точечных файлов лежат в /home/jupyter/project/blends


In [5]:
# private > 0.690
files = {
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv",       # 0.69479
    "cv_weight_v3": "submissions_v3_leak/submission_blend_cv_weight_search_v3.csv",         # 0.69403
    "flaml_automl_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",                 # 0.69348
    "greedy_oof_v3": "submissions_v3_leak/submission_blend_greedy_oof_v3.csv",               # 0.69227
    "lgb_eng_v2": "submissions_v2_leak/submission_lgb_top500_engineered.csv",                # 0.69128
    "prob_equal_v1": "submissions_leak/submission_blend_probability_equal.csv",             # 0.69082
    "log_stacker_v2": "submissions_v2_leak/submission_blend_logistic_stacker_rank.csv",       # 0.69074
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items() if os.path.exists(os.path.join(base_path, f))}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
model_names = list(ranks.keys())
base_df = next(iter(dfs.values()))[[index_col]].copy()

np.random.seed(999)

# ЧАСТЬ 1: БЛЕНДЫ ВООБЩЕ БЕЗ СТЕПЕНЕЙ (Чистые линейные ранги)
print("Генерация блендов БЕЗ СТЕПЕНЕЙ...")
for i in range(1, 21):
    # Выбираем от 4 до 6 элитных моделей
    k = np.random.randint(4, 7)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    # Генерируем веса (сильнейшим моделям - больше веса)
    raw_weights = np.random.uniform(0.1, 1.0, size=k)
    weights = raw_weights / np.sum(raw_weights)
    
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * ranks[m_name]  # Без степеней!
        
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    file_name = f"elite_linear_blend_{i:02d}_models_{k}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)


# ЧАСТЬ 2: ДИВЕРСИФИЦИРОВАННЫЙ ЭЛИТНЫЙ БЛЕНД СО СТЕПЕНЯМИ (Похож на твой старый топ)
print("Генерация элитных блендов со степенями...")
for i in range(1, 21):
    k = np.random.randint(4, len(model_names) + 1)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    weights = np.random.dirichlet(np.ones(k))
    # Держим степень в рамках классических 3.0 - 4.5, как в твоем лучшем решении
    power = round(np.random.uniform(3.0, 4.5), 2)
    
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * (ranks[m_name] ** power)
        
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    file_name = f"elite_power_blend_{i:02d}_models_{k}_pow_{power}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)

print(f"Готово! В папке {output_path} создано 40 очищенных файлов.")


Генерация блендов БЕЗ СТЕПЕНЕЙ...
Генерация элитных блендов со степенями...
Готово! В папке /home/jupyter/project/blends создано 40 очищенных файлов.


In [6]:
# Загружаем файлы, строго соблюдая наличие утечек там, где они были в старом бленде
# Для новых замен берем их лучшие версии из таблицы
files = {
    # Жесткое ядро старого бленда
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv", # в старом бленде шел БЕЗ утечки
    
    # Старая группа поддержки
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml": "submissions_leak/flaml_all_train_fields_0.66582.csv",
    "old_cat": "submissions_leak/old_catboost.csv",
    
    # Новые мощные кандидаты на замену / усиление
    "lgb_gb": "submissions_leak/submission_lightgbm_gradient_boosting.csv",      # 0.68474!
    "logreg": "submissions_leak/submission_logistic_regression.csv",               # 0.68950!
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv"               # 0.69348!
}

index_col = "index"
target_col = "score"

# Предзагрузка
dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items() if os.path.exists(os.path.join(base_path, f))}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = next(iter(dfs.values()))[[index_col]].copy()

# Степень жестко фиксируем около оригинальной (4.5)
power = 4.5

# Вариант 1: Прямая замена old_cat на сильный lgb_gb (Остальное как в старом бленде)
sub1 = base_df.copy()
sub1[target_col] = rankdata(
    0.2525 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.165 * (ranks["tabm"]**power + ranks["lgb_gb"]**power + ranks["flaml"]**power)
) / len(base_df)
sub1.to_csv(os.path.join(output_path, "upgrade_01_cat_to_lgbgb.csv"), index=False)

# Вариант 2: Замена old_cat на взрывную логистическую регрессию
sub2 = base_df.copy()
sub2[target_col] = rankdata(
    0.2525 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.165 * (ranks["tabm"]**power + ranks["logreg"]**power + ranks["flaml"]**power)
) / len(base_df)
sub2.to_csv(os.path.join(output_path, "upgrade_02_cat_to_logreg.csv"), index=False)

# Вариант 3: Замена старого flaml на flaml_v3, а cat на logreg (Максимальный апгрейд поддержки)
sub3 = base_df.copy()
sub3[target_col] = rankdata(
    0.2525 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.165 * (ranks["tabm"]**power + ranks["logreg"]**power + ranks["flaml_v3"]**power)
) / len(base_df)
sub3.to_csv(os.path.join(output_path, "upgrade_03_support_max_boost.csv"), index=False)

# Вариант 4: Добавление logreg шестым элементом с сохранением пропорций ядра
sub4 = base_df.copy()
sub4[target_col] = rankdata(
    0.26 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.12 * (ranks["tabm"]**power + ranks["old_cat"]**power + ranks["flaml"]**power + ranks["logreg"]**power)
) / len(base_df)
sub4.to_csv(os.path.join(output_path, "upgrade_04_add_logreg_6th.csv"), index=False)

# Вариант 5: Добавление lgb_gb шестым элементом
sub5 = base_df.copy()
sub5[target_col] = rankdata(
    0.26 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.12 * (ranks["tabm"]**power + ranks["old_cat"]**power + ranks["flaml"]**power + ranks["lgb_gb"]**power)
) / len(base_df)
sub5.to_csv(os.path.join(output_path, "upgrade_05_add_lgbgb_6th.csv"), index=False)


# Генерируем еще 10 мелких вариаций весов вокруг схемы с заменой Кэтбуста на Логрег/LGB_GB
np.random.seed(42)
for i in range(6, 16):
    # Небольшое качание весов ядра (от 0.22 до 0.28)
    w_core = np.random.uniform(0.22, 0.28)
    # Остаток делим на 3 модели поддержки (tabm, flaml_v3 и либо logreg, либо lgb_gb)
    w_supp = (1.0 - (w_core * 2)) / 3
    
    chosen_cat_repl = "logreg" if i % 2 == 0 else "lgb_gb"
    
    final_rank = (
        w_core * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) +
        w_supp * (ranks["tabm"]**power + ranks["flaml_v3"]**power + ranks[chosen_cat_repl]**power)
    )
    
    sub_loop = base_df.copy()
    sub_loop[target_col] = rankdata(final_rank) / len(final_rank)
    sub_loop.to_csv(os.path.join(output_path, f"upgrade_{i:02d}_loop.csv"), index=False)

print(f"Готово! В папке {output_path} создано 15 точечных апгрейдов твоего лучшего бленда.")


Готово! В папке /home/jupyter/project/blends создано 15 точечных апгрейдов твоего лучшего бленда.


In [7]:
# Полный боекомплект лучших проверенных компонентов
files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = next(iter(dfs.values()))[[index_col]].copy()

np.random.seed(2026) # Заряжаем сид на прорыв

# --- СТРАТЕГИЯ 1: ПЕРЕВЕРНУТОЕ ЯДРО (12 файлов) ---
# Даем моделям поддержки (особенно логистике и фламлу) доминирующие веса
for i in range(1, 13):
    w_lgb_eng = np.random.uniform(0.14, 0.18)
    w_lgb_dart = np.random.uniform(0.14, 0.18)
    
    # Распределяем оставшийся большой вес между поддержкой
    w_left = 1.0 - (w_lgb_eng + w_lgb_dart)
    raw_supp = np.random.dirichlet(np.ones(3)) # для tabm, flaml_v3, logreg
    w_tabm, w_flaml, w_logreg = raw_supp * w_left
    
    p = round(np.random.uniform(3.5, 5.0), 2)
    
    final_rank = (
        w_lgb_eng * (ranks["lgb_eng"]**p) +
        w_lgb_dart * (ranks["lgb_dart"]**p) +
        w_tabm * (ranks["tabm"]**p) +
        w_flaml * (ranks["flaml_v3"]**p) +
        w_logreg * (ranks["logreg"]**p)
    )
    
    sub = base_df.copy()
    sub[target_col] = rankdata(final_rank) / len(final_rank)
    sub.to_csv(os.path.join(output_path, f"strat1_inverted_core_{i:02d}_p_{p}.csv"), index=False)


# --- СТРАТЕГИЯ 2: ДВОЙНОЕ ЛОГИСТИЧЕСКОЕ УСИЛЕНИЕ (10 файлов) ---
# Смешиваем 6 моделей, где logreg и log_stacker_v3 получают повышенные веса
for i in range(1, 11):
    w_core = np.random.uniform(0.18, 0.22) # lgb_eng и lgb_dart
    w_log_heavy = np.random.uniform(0.18, 0.23) # logreg и log_stacker_v3
    
    w_rest = (1.0 - (w_core * 2) - (w_log_heavy * 2)) / 2 # на tabm и flaml_v3
    
    p = round(np.random.uniform(4.0, 4.8), 2)
    
    final_rank = (
        w_core * (ranks["lgb_eng"]**p + ranks["lgb_dart"]**p) +
        w_log_heavy * (ranks["logreg"]**p + ranks["log_stacker_v3"]**p) +
        w_rest * (ranks["tabm"]**p + ranks["flaml_v3"]**p)
    )
    
    sub = base_df.copy()
    sub[target_col] = rankdata(final_rank) / len(final_rank)
    sub.to_csv(os.path.join(output_path, f"strat2_double_logistic_{i:02d}_p_{p}.csv"), index=False)


# --- СТРАТЕГИЯ 3: АГРЕССИВНЫЙ POWER-СВИНГ (10 файлов) ---
# Фиксируем пропорции лучшего upgrade_10, но бьем по экстремальным степеням
w_core_fixed = 0.225
w_supp_fixed = 0.183
extreme_powers = [1.5, 2.0, 2.5, 3.0, 5.2, 5.5, 6.0, 6.5, 7.0, 8.0]

for idx, p in enumerate(extreme_powers, 1):
    final_rank = (
        w_core_fixed * (ranks["lgb_eng"]**p + ranks["lgb_dart"]**p) +
        w_supp_fixed * (ranks["tabm"]**p + ranks["flaml_v3"]**p + ranks["logreg"]**p)
    )
    sub = base_df.copy()
    sub[target_col] = rankdata(final_rank) / len(final_rank)
    sub.to_csv(os.path.join(output_path, f"strat3_power_swing_{idx:02d}_p_{p}.csv"), index=False)

print(f"Пайплайн завершен! Ровно 32 агрессивных снаряда готовы к отправке в {output_path}")

Пайплайн завершен! Ровно 32 агрессивных снаряда готовы к отправке в /home/jupyter/project/blends


In [8]:
files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = next(iter(dfs.values()))[[index_col]].copy()

def save_sub(final_rank, filename):
    final_score = rankdata(final_rank) / len(final_rank)
    sub = base_df.copy()
    sub[target_col] = final_score
    sub.to_csv(os.path.join(output_path, filename), index=False)

# ---- ШОТ 1: Уклон в жесткое бустинг-ядро (повышаем резкость) ----
p1 = 4.15
rank_1 = (
    0.23 * (ranks["lgb_eng"]**p1 + ranks["lgb_dart"]**p1) +
    0.18 * (ranks["logreg"]**p1 + ranks["log_stacker_v3"]**p1) +
    0.09 * (ranks["tabm"]**p1 + ranks["flaml_v3"]**p1)
)
save_sub(rank_1, "final_shot_01_heavy_core_p4.15.csv")

# ---- ШОТ 2: Уклон в двойную логистику (калибровка на максимум) ----
p2 = 4.10
rank_2 = (
    0.18 * (ranks["lgb_eng"]**p2 + ranks["lgb_dart"]**p2) +
    0.23 * (ranks["logreg"]**p2 + ranks["log_stacker_v3"]**p2) +
    0.09 * (ranks["tabm"]**p2 + ranks["flaml_v3"]**p2)
)
save_sub(rank_2, "final_shot_02_heavy_log_p4.10.csv")

# ---- ШОТ 3: Золотая середина с пониженной степенью ----
p3 = 3.95
rank_3 = (
    0.20 * (ranks["lgb_eng"]**p3 + ranks["lgb_dart"]**p3) +
    0.20 * (ranks["logreg"]**p3 + ranks["log_stacker_v3"]**p3) +
    0.10 * (ranks["tabm"]**p3 + ranks["flaml_v3"]**p3)
)
save_sub(rank_3, "final_shot_03_balanced_p3.95.csv")

# ---- ШОТ 4: Усиление за счет FLAML_v3 (взято из паттернов Strategy 1) ----
p4 = 4.05
rank_4 = (
    0.19 * (ranks["lgb_eng"]**p4 + ranks["lgb_dart"]**p4) +
    0.19 * (ranks["logreg"]**p4 + ranks["log_stacker_v3"]**p4) +
    0.07 * (ranks["tabm"]**p4) + 0.17 * (ranks["flaml_v3"]**p4)
)
save_sub(rank_4, "final_shot_04_flaml_boost_p4.05.csv")

# ---- ШОТ 5: Мягкое сглаживание (минимальная степень для борьбы с переобучением) ----
p5 = 3.85
rank_5 = (
    0.21 * (ranks["lgb_eng"]**p5 + ranks["lgb_dart"]**p5) +
    0.21 * (ranks["logreg"]**p5 + ranks["log_stacker_v3"]**p5) +
    0.08 * (ranks["tabm"]**p5 + ranks["flaml_v3"]**p5)
)
save_sub(rank_5, "final_shot_05_smooth_p3.85.csv")

print(f"Все 5 элитных финальных сабмитов лежат в {output_path}!")


Все 5 элитных финальных сабмитов лежат в /home/jupyter/project/blends!


### Попробуем докрутить лучший сабмит `strat2_double_logistic_02_p_4.13.csv` (private score **0.71454**)

In [2]:
# 1. Путь к лучшему сабмиту
best_submit_path = "/home/jupyter/project/blends/strat2_double_logistic_02_p_4.13.csv"

# 2. Список папок для сканирования
directories = [
    "/home/jupyter/project/submissions",
    "/home/jupyter/project/submissions_leak",
    "/home/jupyter/project/submissions_v2",
    "/home/jupyter/project/submissions_v2_leak",
    "/home/jupyter/project/submissions_v3",
    "/home/jupyter/project/submissions_v3_leak"
]

# Загружаем лучший сабмит и сортируем по index для гарантированного совпадения строк
print("Загрузка лучшего сабмита...")
best_df = pd.read_csv(best_submit_path).sort_values("index").reset_index(drop=True)

results = []

# 3. Обход всех папок и файлов
for folder in directories:
    if not os.path.exists(folder):
        print(f"Предупреждение: Папка {folder} не существует. Пропускаем.")
        continue
        
    print(f"Обработка папки: {folder}")
    files = [f for f in os.listdir(folder) if f.endswith('.csv')]
    
    for file in tqdm(files):
        file_path = os.path.join(folder, file)
        
        try:
            # Читаем текущий сабмит
            current_df = pd.read_csv(file_path).sort_values("index").reset_index(drop=True)
            
            # Проверяем, что размеры и индексы совпадают
            if len(current_df) != len(best_df):
                print(f"Размер файла {file} ({len(current_df)}) не совпадает с лучшим ({len(best_df)}). Пропускаем.")
                continue
                
            # Считаем ранговую корреляцию Спирмена
            correlation = best_df["score"].corr(current_df["score"], method="spearman")
            
            results.append({
                "folder": folder,
                "filename": file,
                "full_path": file_path,
                "spearman_corr": correlation
            })
            
        except Exception as e:
            print(f"Ошибка при обработке файла {file}: {e}")

# 4. Создаем итоговый датафрейм
df_corr = pd.DataFrame(results)

# Сортируем по возрастанию корреляции (самые уникальные будут вверху)
df_corr = df_corr.sort_values("spearman_corr").reset_index(drop=True)

# Выводим топ-20 самых раскоррелированных файлов
print("\nТоп-30 файлов с наименьшей корреляцией к лучшему сабмиту:")
print(df_corr.head(30)[["filename", "spearman_corr"]])

# Сохраняем результат, чтобы не пересчитывать заново
# df_corr.to_csv("sub_correlations.csv", index=False)


Загрузка лучшего сабмита...
Обработка папки: /home/jupyter/project/submissions


100%|██████████| 13/13 [00:01<00:00,  9.88it/s]


Обработка папки: /home/jupyter/project/submissions_leak


100%|██████████| 13/13 [00:01<00:00, 10.31it/s]


Обработка папки: /home/jupyter/project/submissions_v2


...

Обработка папки: /home/jupyter/project/submissions_v3


100%|██████████| 24/24 [00:02<00:00,  9.77it/s]


Обработка папки: /home/jupyter/project/submissions_v3_leak


100%|██████████| 24/24 [00:02<00:00, 10.57it/s]


Топ-30 файлов с наименьшей корреляцией к лучшему сабмиту:
                                             filename  spearman_corr
0                   submission_mlp_neural_network.csv       0.413635
1                   submission_mlp_neural_network.csv       0.448288
2                submission_sgd_modified_huber_v3.csv       0.532021
3                            submission_lgb_rf_v3.csv       0.569576
4                submission_sgd_modified_huber_v3.csv       0.582582
5                            submission_lgb_rf_v3.csv       0.585914
6                              rnn_top300_0.62366.csv       0.644862
7                      submission_mlp_quantile_v3.csv       0.650289
8                              rnn_top300_0.62366.csv       0.654117
9                       submission_extra_trees_v3.csv       0.662915
10                     submission_mlp_quantile_v3.csv       0.664857
11                      submission_extra_trees_v3.csv       0.673819
12                       submission_random_f

In [5]:
import os
import pandas as pd
import numpy as np
from scipy.stats import rankdata

# Базовые пути
base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/all_experiments"
os.makedirs(output_path, exist_ok=True)

# 1. Загружаем и готовим базовое проверенное ядро (6 моделей)
files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

dfs = {name: pd.read_csv(os.path.join(base_path, f)).sort_values("index").reset_index(drop=True) for name, f in files.items()}
ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

# Константы вашего лучшего сабмита
w_core = 0.20
w_log_heavy = 0.21
w_rest = 0.09
p = 4.13

core_rank_sum = (
    w_core * (ranks["lgb_eng"]**p + ranks["lgb_dart"]**p) +
    w_log_heavy * (ranks["logreg"]**p + ranks["log_stacker_v3"]**p) +
    w_rest * (ranks["tabm"]**p + ranks["flaml_v3"]**p)
)

w_candidate = 0.07  
w_base = 0.93

# 2. Жёстко ограничиваем цикл первыми 30 строками из df_corr
# .head(30) гарантирует, что мы берем только ваш топ раскоррелированных
df_corr_top30 = df_corr.head(30)

print(f"Запуск генерации СТРОГО {len(df_corr_top30)} сабмитов...")

for idx, row in df_corr_top30.iterrows():
    file_path = row["full_path"]
    filename = row["filename"].replace(".csv", "")
    corr_val = round(row["spearman_corr"], 3)
    
    try:
        cand_df = pd.read_csv(file_path).sort_values("index").reset_index(drop=True)
        cand_rank = rankdata(cand_df["score"]) / len(cand_df)
        
        final_rank = (w_base * core_rank_sum) + (w_candidate * (cand_rank ** p))
        
        sub = base_df.copy()
        sub["score"] = rankdata(final_rank) / len(final_rank)
        
        # Формируем имя файла
        sub_name = f"{idx:02d}__{filename}__corr_{corr_val}.csv"
        sub.to_csv(os.path.join(output_path, sub_name), index=False)
        print(f"Создан [{idx+1}/30]: {sub_name}")
        
    except Exception as e:
        print(f"Ошибка при обработке {filename}: {e}")

print(f"\nГотово! В папке ровно {len(df_corr_top30)} файлов.")


Запуск генерации СТРОГО 30 сабмитов...
Создан [1/30]: 00__submission_mlp_neural_network__corr_0.414.csv
Создан [2/30]: 01__submission_mlp_neural_network__corr_0.448.csv
Создан [3/30]: 02__submission_sgd_modified_huber_v3__corr_0.532.csv
Создан [4/30]: 03__submission_lgb_rf_v3__corr_0.57.csv
Создан [5/30]: 04__submission_sgd_modified_huber_v3__corr_0.583.csv
Создан [6/30]: 05__submission_lgb_rf_v3__corr_0.586.csv
Создан [7/30]: 06__rnn_top300_0.62366__corr_0.645.csv
Создан [8/30]: 07__submission_mlp_quantile_v3__corr_0.65.csv
Создан [9/30]: 08__rnn_top300_0.62366__corr_0.654.csv
Создан [10/30]: 09__submission_extra_trees_v3__corr_0.663.csv
Создан [11/30]: 10__submission_mlp_quantile_v3__corr_0.665.csv
Создан [12/30]: 11__submission_extra_trees_v3__corr_0.674.csv
Создан [13/30]: 12__submission_random_forest__corr_0.695.csv
Создан [14/30]: 13__submission_random_forest__corr_0.699.csv
Создан [15/30]: 14__submission_extratrees_v2_pca_svd_stack_fullfit__corr_0.7.csv
Создан [16/30]: 15__tabm_

In [9]:
import os
import pandas as pd
import numpy as np
from scipy.stats import rankdata

# Базовые пути
base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/final_private_search"
os.makedirs(output_path, exist_ok=True)

# 1. Ваше проверенное базовое ядро (6 моделей)
core_files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

# 2. Новые Private-лидеры 
new_files = {
    "mlp_q10": "submissions_v3_leak/submission_mlp_quantile_v3.csv",             
    "flaml_all27": "submissions_leak/flaml_all_train_fields_0.66582.csv", 
    "et_pca29": "submissions_v2_leak/submission_extratrees_v2_pca_svd_stack.csv"  
}

all_files = {**core_files, **new_files}
dfs = {}

# Загрузка
for name, f in all_files.items():
    full_path = os.path.join(base_path, f)
    dfs[name] = pd.read_csv(full_path).sort_values("index").reset_index(drop=True)

ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

np.random.seed(2026)
N_SUBMITS = 25

print(f"Запуск генерации СТРОГО {N_SUBMITS} сабмитов...")

for i in range(1, N_SUBMITS + 1):
    # Генерируем "сырые" веса в нужных вам пропорциях (ядро тяжелее, новички легче)
    raw_weights = {
        "lgb_eng": np.random.uniform(0.18, 0.22),
        "lgb_dart": np.random.uniform(0.18, 0.22),
        "logreg": np.random.uniform(0.18, 0.23),
        "log_stacker_v3": np.random.uniform(0.18, 0.23),
        "tabm": np.random.uniform(0.07, 0.10),
        "flaml_v3": np.random.uniform(0.07, 0.10),
        # Новые Private-компоненты
        "flaml_all27": np.random.uniform(0.04, 0.08),
        "mlp_q10": np.random.uniform(0.03, 0.06),
        "et_pca29": np.random.uniform(0.02, 0.05)
    }
    
    # Жесткая нормализация: делим каждый вес на общую сумму. Сумма всегда станет 1.0!
    total_sum = sum(raw_weights.values())
    w = {name: val / total_sum for name, val in raw_weights.items()}
    
    p = round(np.random.uniform(4.0, 4.5), 2)
    
    # Считаем финальный ранговый бленд по абсолютно всем 9 моделям
    final_rank = (
        w["lgb_eng"] * (ranks["lgb_eng"]**p) +
        w["lgb_dart"] * (ranks["lgb_dart"]**p) +
        w["logreg"] * (ranks["logreg"]**p) +
        w["log_stacker_v3"] * (ranks["log_stacker_v3"]**p) +
        w["tabm"] * (ranks["tabm"]**p) +
        w["flaml_v3"] * (ranks["flaml_v3"]**p) +
        w["flaml_all27"] * (ranks["flaml_all27"]**p) +
        w["mlp_q10"] * (ranks["mlp_q10"]**p) +
        w["et_pca29"] * (ranks["et_pca29"]**p)
    )
    
    sub = base_df.copy()
    sub["score"] = rankdata(final_rank) / len(final_rank)
    
    # Имя файла с весами ключевых Private-моделей
    sub_name = f"priv_blend_{i:02d}__p_{p}__f27_{w['flaml_all27']:.3f}__m10_{w['mlp_q10']:.3f}.csv"
    sub.to_csv(os.path.join(output_path, sub_name), index=False)
    print(f"Успешно создан [{i:02d}/{N_SUBMITS}]: {sub_name}")

print(f"\nГотово! Создано ровно {N_SUBMITS} файлов.")


Запуск генерации СТРОГО 25 сабмитов...
Успешно создан [01/25]: priv_blend_01__p_4.39__f27_0.043__m10_0.051.csv
Успешно создан [02/25]: priv_blend_02__p_4.46__f27_0.056__m10_0.037.csv
Успешно создан [03/25]: priv_blend_03__p_4.34__f27_0.043__m10_0.030.csv
Успешно создан [04/25]: priv_blend_04__p_4.23__f27_0.056__m10_0.040.csv
Успешно создан [05/25]: priv_blend_05__p_4.04__f27_0.061__m10_0.047.csv
Успешно создан [06/25]: priv_blend_06__p_4.33__f27_0.072__m10_0.040.csv
Успешно создан [07/25]: priv_blend_07__p_4.17__f27_0.044__m10_0.051.csv
Успешно создан [08/25]: priv_blend_08__p_4.38__f27_0.040__m10_0.032.csv
Успешно создан [09/25]: priv_blend_09__p_4.05__f27_0.043__m10_0.047.csv
Успешно создан [10/25]: priv_blend_10__p_4.1__f27_0.065__m10_0.041.csv
Успешно создан [11/25]: priv_blend_11__p_4.23__f27_0.056__m10_0.043.csv
Успешно создан [12/25]: priv_blend_12__p_4.35__f27_0.049__m10_0.030.csv
Успешно создан [13/25]: priv_blend_13__p_4.49__f27_0.044__m10_0.035.csv
Успешно создан [14/25]: pr

In [10]:
import os
import pandas as pd
import numpy as np
from scipy.stats import rankdata

base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/heavy_private_attack"
os.makedirs(output_path, exist_ok=True)

# 1. Базовое ядро (6 моделей)
core_files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

# 2. Новые Private-лидеры (ИСПРАВЛЕН путь для flaml_all27)
new_files = {
    "mlp_q10": "submissions_v3_leak/submission_mlp_quantile_v3.csv",             
    "flaml_all27": "submissions_leak/flaml_all_train_fields_0.66582.csv", # Теперь строго submissions_leak
    "et_pca29": "submissions_v2_leak/submission_extratrees_v2_pca_svd_stack.csv"  
}

all_files = {**core_files, **new_files}
dfs = {}

for name, f in all_files.items():
    full_path = os.path.join(base_path, f)
    dfs[name] = pd.read_csv(full_path).sort_values("index").reset_index(drop=True)

ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

np.random.seed(999) # Меняем сид на удачу
N_SUBMITS = 15

print(f"Запуск АГРЕССИВНОЙ генерации {N_SUBMITS} сабмитов...")

for i in range(1, N_SUBMITS + 1):
    # Генерируем тяжелые веса для новых Private-лидеров
    # Суммарно этот блок будет забирать от 20% до 35% всего ансамбля
    w_flaml27 = np.random.uniform(0.10, 0.18) # Главный драйвер скора
    w_mlp10 = np.random.uniform(0.06, 0.10)   # Дает раскорреляцию
    w_et29 = np.random.uniform(0.04, 0.08)    # Стабилизирует дерево
    
    new_total = w_flaml27 + w_mlp10 + w_et29
    
    # Оставшийся вес (65% - 80%) отдаем базовому проверенному ядру
    core_leftover = 1.0 - new_total
    
    # Распределяем остаток внутри ядра с сохранением исторических пропорций
    raw_core = {
        "lgb_eng": np.random.uniform(0.18, 0.22),
        "lgb_dart": np.random.uniform(0.18, 0.22),
        "logreg": np.random.uniform(0.18, 0.23),
        "log_stacker_v3": np.random.uniform(0.18, 0.23),
        "tabm": np.random.uniform(0.07, 0.10),
        "flaml_v3": np.random.uniform(0.07, 0.10)
    }
    
    core_sum = sum(raw_core.values())
    w = {name: (val / core_sum) * core_leftover for name, val in raw_core.items()}
    
    # Добавляем веса новых моделей в общий словарь весов
    w["flaml_all27"] = w_flaml27
    w["mlp_q10"] = w_mlp10
    w["et_pca29"] = w_et29
    
    # Пробуем чуть более гибкий шаг по степени p
    p = round(np.random.uniform(3.8, 4.6), 2)
    
    # Считаем бленд
    final_rank = (
        w["lgb_eng"] * (ranks["lgb_eng"]**p) +
        w["lgb_dart"] * (ranks["lgb_dart"]**p) +
        w["logreg"] * (ranks["logreg"]**p) +
        w["log_stacker_v3"] * (ranks["log_stacker_v3"]**p) +
        w["tabm"] * (ranks["tabm"]**p) +
        w["flaml_v3"] * (ranks["flaml_v3"]**p) +
        w["flaml_all27"] * (ranks["flaml_all27"]**p) +
        w["mlp_q10"] * (ranks["mlp_q10"]**p) +
        w["et_pca29"] * (ranks["et_pca29"]**p)
    )
    
    sub = base_df.copy()
    sub["score"] = rankdata(final_rank) / len(final_rank)
    
    sub_name = f"heavy_blend_{i:02d}__p_{p}__f27_{w_flaml27:.3f}__m10_{w_mlp10:.3f}.csv"
    sub.to_csv(os.path.join(output_path, sub_name), index=False)
    print(f"Создан агрессивный сабмит [{i:02d}/{N_SUBMITS}]: p={p}, New_models_weight={new_total:.2f}")

print(f"\nГотово!")


Запуск АГРЕССИВНОЙ генерации 15 сабмитов...
Создан агрессивный сабмит [01/15]: p=4.36, New_models_weight=0.29
Создан агрессивный сабмит [02/15]: p=4.18, New_models_weight=0.28
Создан агрессивный сабмит [03/15]: p=4.24, New_models_weight=0.26
Создан агрессивный сабмит [04/15]: p=4.47, New_models_weight=0.28
Создан агрессивный сабмит [05/15]: p=3.95, New_models_weight=0.35
Создан агрессивный сабмит [06/15]: p=4.52, New_models_weight=0.32
Создан агрессивный сабмит [07/15]: p=3.96, New_models_weight=0.32
Создан агрессивный сабмит [08/15]: p=4.16, New_models_weight=0.33
Создан агрессивный сабмит [09/15]: p=3.95, New_models_weight=0.28
Создан агрессивный сабмит [10/15]: p=4.09, New_models_weight=0.33
Создан агрессивный сабмит [11/15]: p=4.5, New_models_weight=0.31
Создан агрессивный сабмит [12/15]: p=4.33, New_models_weight=0.31
Создан агрессивный сабмит [13/15]: p=4.58, New_models_weight=0.31
Создан агрессивный сабмит [14/15]: p=4.23, New_models_weight=0.22
Создан агрессивный сабмит [15/15]

In [11]:
# 1. Путь к лучшему сабмиту
best_submit_path = "/home/jupyter/project/blends/strat2_double_logistic_02_p_4.13.csv"
best_df = pd.read_csv(best_submit_path).sort_values("index").reset_index(drop=True)

# 2. Пары файлов для сравнения (с утечкой vs без утечки)
compare_files = {
    "MLP Quantile (С утечкой)": "submissions_v3_leak/submission_mlp_quantile_v3.csv",
    "MLP Quantile (БЕЗ утечки)": "submissions_v3/submission_mlp_quantile_v3.csv",
    
    "FLAML All Fields (С утечкой)": "submissions_leak/flaml_all_train_fields_0.66582.csv",
    "FLAML All Fields (БЕЗ утечки)": "submissions/flaml_all_train_fields_0.66582.csv",
    
    "ExtraTrees PCA (С утечкой)": "submissions_v2_leak/submission_extratrees_v2_pca_svd_stack.csv",
    "ExtraTrees PCA (БЕЗ утечки)": "submissions_v2/submission_extratrees_v2_pca_svd_stack.csv"
}

print("=== Сравнение корреляции Спирмена с лучшим сабмитом ===")
results = []

for name, rel_path in compare_files.items():
    full_path = os.path.join("/home/jupyter/project", rel_path)
    
    if not os.path.exists(full_path):
        print(f"Файл не найден: {rel_path}")
        continue
        
    df = pd.read_csv(full_path).sort_values("index").reset_index(drop=True)
    corr = best_df["score"].corr(df["score"], method="spearman")
    
    results.append({"Модель": name, "Корреляция": round(corr, 4)})

# Вывод красивого датафрейма
df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))


=== Сравнение корреляции Спирмена с лучшим сабмитом ===
                       Модель  Корреляция
     MLP Quantile (С утечкой)      0.6649
    MLP Quantile (БЕЗ утечки)      0.6503
 FLAML All Fields (С утечкой)      0.7687
FLAML All Fields (БЕЗ утечки)      0.7807
   ExtraTrees PCA (С утечкой)      0.7589
  ExtraTrees PCA (БЕЗ утечки)      0.7749


In [14]:
# 1. Путь к вашему исторически лучшему сабмиту
best_submit_path = "/home/jupyter/project/blends/strat2_double_logistic_02_p_4.13.csv"
best_df = pd.read_csv(best_submit_path).sort_values("index").reset_index(drop=True)

# 2. Пары файлов для проверки (с утечкой из папок _leak vs чистые версии)
compare_low_corr = {
    "SGD Huber (С утечкой)": "submissions_v3_leak/submission_sgd_modified_huber_v3.csv",
    "SGD Huber (БЕЗ утечки)": "submissions_v3/submission_sgd_modified_huber_v3.csv",
    
    "RNN (С утечкой)": "submissions_leak/rnn_top300_0.62366.csv",
    "RNN (БЕЗ утечки)": "submissions/rnn_top300_0.62366.csv"
}

print("=== Сравнение корреляции Спирмена низкокоррелированных моделей ===")
results = []

for name, rel_path in compare_low_corr.items():
    full_path = os.path.join("/home/jupyter/project", rel_path)
    
    if not os.path.exists(full_path):
        print(f"Предупреждение: Файл не найден по пути {rel_path}. Пропустите или поправьте папку.")
        continue
        
    df = pd.read_csv(full_path).sort_values("index").reset_index(drop=True)
    corr = best_df["score"].corr(df["score"], method="spearman")
    
    results.append({"Модель": name, "Корреляция Спирмена": round(corr, 4)})

# Выводим итоговую таблицу
df_res = pd.DataFrame(results)
print("\n", df_res.to_string(index=False))


=== Сравнение корреляции Спирмена низкокоррелированных моделей ===

                 Модель  Корреляция Спирмена
 SGD Huber (С утечкой)               0.5826
SGD Huber (БЕЗ утечки)               0.5320
       RNN (С утечкой)               0.6541
      RNN (БЕЗ утечки)               0.6449


In [15]:
base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/low_corr_regularization"
os.makedirs(output_path, exist_ok=True)

# 1. Ваше проверенное базовое ядро (6 моделей)
core_files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

# 2. Дополнительные компоненты (Строго выверенные папки leak/clean)
extra_files = {
    "mlp_q10": "submissions_v3/submission_mlp_quantile_v3.csv",             # Чистая (Лучший Private)
    "flaml_all27": "submissions/flaml_all_train_fields_0.66582.csv",         # Чистая (Высокая corr)
    "et_pca29": "submissions_v2/submission_extratrees_v2_pca_svd_stack.csv",  # Чистая (Высокая corr)
    "sgd_huber": "submissions_v3_leak/submission_sgd_modified_huber_v3.csv", # С утечкой (Выше corr)
    "rnn": "submissions_leak/rnn_top300_0.62366.csv"                      # С утечкой (Выше corr)
}

all_files = {**core_files, **extra_files}
dfs = {}

for name, f in all_files.items():
    full_path = os.path.join(base_path, f)
    dfs[name] = pd.read_csv(full_path).sort_values("index").reset_index(drop=True)

ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

np.random.seed(888) # Сид на глубокую регуляризацию
N_SUBMITS = 20

print(f"Запуск генерации {N_SUBMITS} диверсифицированных сабмитов...")

for i in range(1, N_SUBMITS + 1):
    # Микро-веса для сильной раскорреляции (SGD Huber и RNN с утечкой)
    w_sgd = np.random.uniform(0.01, 0.025)
    w_rnn = np.random.uniform(0.015, 0.03)
    
    # Веса для стабильных чистых компонентов
    w_flaml27 = np.random.uniform(0.03, 0.06)
    w_mlp10 = np.random.uniform(0.02, 0.04)
    w_et29 = np.random.uniform(0.015, 0.035)
    
    extra_total = w_sgd + w_rnn + w_flaml27 + w_mlp10 + w_et29
    core_leftover = 1.0 - extra_total
    
    # Пропорции базового ядра
    raw_core = {
        "lgb_eng": np.random.uniform(0.18, 0.22),
        "lgb_dart": np.random.uniform(0.18, 0.22),
        "logreg": np.random.uniform(0.18, 0.23),
        "log_stacker_v3": np.random.uniform(0.18, 0.23),
        "tabm": np.random.uniform(0.07, 0.10),
        "flaml_v3": np.random.uniform(0.07, 0.10)
    }
    
    core_sum = sum(raw_core.values())
    w = {name: (val / core_sum) * core_leftover for name, val in raw_core.items()}
    
    # Степень нелинейности p
    p = round(np.random.uniform(4.1, 4.4), 2)
    
    # Считаем финальный бленд рангов из 11 моделей
    final_rank = (
        w["lgb_eng"] * (ranks["lgb_eng"]**p) +
        w["lgb_dart"] * (ranks["lgb_dart"]**p) +
        w["logreg"] * (ranks["logreg"]**p) +
        w["log_stacker_v3"] * (ranks["log_stacker_v3"]**p) +
        w["tabm"] * (ranks["tabm"]**p) +
        w["flaml_v3"] * (ranks["flaml_v3"]**p) +
        w_flaml27 * (ranks["flaml_all27"]**p) +
        w_mlp10 * (ranks["mlp_q10"]**p) +
        w_et29 * (ranks["et_pca29"]**p) +
        w_sgd * (ranks["sgd_huber"]**p) +
        w_rnn * (ranks["rnn"]**p)
    )
    
    sub = base_df.copy()
    sub["score"] = rankdata(final_rank) / len(final_rank)
    
    sub_name = f"div_blend_{i:02d}__p_{p}__sgd_{w_sgd:.3f}__rnn_{w_rnn:.3f}.csv"
    sub.to_csv(os.path.join(output_path, sub_name), index=False)
    print(f"Создан сабмит [{i:02d}/{N_SUBMITS}]: p={p}, Свободный вес ядра={core_leftover:.2f}")

print(f"\nВсе 20 файлов успешно сохранены в: {output_path}")


Запуск генерации 20 диверсифицированных сабмитов...
Создан сабмит [01/20]: p=4.17, Свободный вес ядра=0.85
Создан сабмит [02/20]: p=4.31, Свободный вес ядра=0.85
Создан сабмит [03/20]: p=4.36, Свободный вес ядра=0.89
Создан сабмит [04/20]: p=4.11, Свободный вес ядра=0.86
Создан сабмит [05/20]: p=4.14, Свободный вес ядра=0.87
Создан сабмит [06/20]: p=4.37, Свободный вес ядра=0.84
Создан сабмит [07/20]: p=4.16, Свободный вес ядра=0.86
Создан сабмит [08/20]: p=4.11, Свободный вес ядра=0.87
Создан сабмит [09/20]: p=4.33, Свободный вес ядра=0.86
Создан сабмит [10/20]: p=4.23, Свободный вес ядра=0.86
Создан сабмит [11/20]: p=4.17, Свободный вес ядра=0.87
Создан сабмит [12/20]: p=4.16, Свободный вес ядра=0.87
Создан сабмит [13/20]: p=4.12, Свободный вес ядра=0.84
Создан сабмит [14/20]: p=4.4, Свободный вес ядра=0.87
Создан сабмит [15/20]: p=4.13, Свободный вес ядра=0.87
Создан сабмит [16/20]: p=4.29, Свободный вес ядра=0.86
Создан сабмит [17/20]: p=4.22, Свободный вес ядра=0.86
Создан сабмит 

In [16]:
base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/surgical_flaml_attack"
os.makedirs(output_path, exist_ok=True)

# 1. Ваше проверенное базовое ядро (6 моделей)
core_files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

# 2. Единственный подтвержденный лидер
flaml_path = "submissions_leak/flaml_all_train_fields_0.66582.csv"

dfs = {name: pd.read_csv(os.path.join(base_path, f)).sort_values("index").reset_index(drop=True) for name, f in core_files.items()}
dfs["flaml_all27"] = pd.read_csv(os.path.join(base_path, flaml_path)).sort_values("index").reset_index(drop=True)

ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

np.random.seed(42) # Сид на чистый точечный фокус
N_SUBMITS = 12

print(f"Запуск хирургической генерации {N_SUBMITS} сабмитов...")

for i in range(1, N_SUBMITS + 1):
    # Задаем строго контролируемый вес для flaml_all27 (от 5% до 13%)
    w_flaml27 = 0.04 + (i * 0.008) 
    core_leftover = 1.0 - w_flaml27
    
    # Веса ядра генерируем в узком коридоре вокруг вашего лучшего исторического решения
    raw_core = {
        "lgb_eng": np.random.uniform(0.19, 0.21),
        "lgb_dart": np.random.uniform(0.19, 0.21),
        "logreg": np.random.uniform(0.19, 0.22),
        "log_stacker_v3": np.random.uniform(0.19, 0.22),
        "tabm": np.random.uniform(0.08, 0.095),
        "flaml_v3": np.random.uniform(0.08, 0.095)
    }
    
    core_sum = sum(raw_core.values())
    w = {name: (val / core_sum) * core_leftover for name, val in raw_core.items()}
    
    # Степень p фиксируем около вашего идеального значения 4.13
    p = round(np.random.uniform(4.10, 4.25), 2)
    
    # Считаем бленд: ядро + одна целевая модель
    final_rank = (
        w["lgb_eng"] * (ranks["lgb_eng"]**p) +
        w["lgb_dart"] * (ranks["lgb_dart"]**p) +
        w["logreg"] * (ranks["logreg"]**p) +
        w["log_stacker_v3"] * (ranks["log_stacker_v3"]**p) +
        w["tabm"] * (ranks["tabm"]**p) +
        w["flaml_v3"] * (ranks["flaml_v3"]**p) +
        w_flaml27 * (ranks["flaml_all27"]**p)
    )
    
    sub = base_df.copy()
    sub["score"] = rankdata(final_rank) / len(final_rank)
    
    sub_name = f"surgical_blend_{i:02d}__p_{p}__flamlweight_{w_flaml27:.3f}.csv"
    sub.to_csv(os.path.join(output_path, sub_name), index=False)
    print(f"Создан сабмит [{i:02d}/{N_SUBMITS}]: вес FLAML = {w_flaml27:.3f}, p = {p}")

print(f"\nВсе файлы сохранены в: {output_path}")


Запуск хирургической генерации 12 сабмитов...
Создан сабмит [01/12]: вес FLAML = 0.048, p = 4.11
Создан сабмит [02/12]: вес FLAML = 0.056, p = 4.13
Создан сабмит [03/12]: вес FLAML = 0.064, p = 4.19
Создан сабмит [04/12]: вес FLAML = 0.072, p = 4.18
Создан сабмит [05/12]: вес FLAML = 0.080, p = 4.24
Создан сабмит [06/12]: вес FLAML = 0.088, p = 4.17
Создан сабмит [07/12]: вес FLAML = 0.096, p = 4.18
Создан сабмит [08/12]: вес FLAML = 0.104, p = 4.24
Создан сабмит [09/12]: вес FLAML = 0.112, p = 4.22
Создан сабмит [10/12]: вес FLAML = 0.120, p = 4.25
Создан сабмит [11/12]: вес FLAML = 0.128, p = 4.22
Создан сабмит [12/12]: вес FLAML = 0.136, p = 4.11

Все файлы сохранены в: /home/jupyter/project/blends/surgical_flaml_attack


In [17]:
base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/final_all_in_attack"
os.makedirs(output_path, exist_ok=True)

# 1. Базовое ядро (6 моделей)
core_files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

# 2. Два сильных чистых союзника
flaml_path = "submissions_leak/flaml_all_train_fields_0.66582.csv"
mlp_path = "submissions_v3/submission_mlp_quantile_v3.csv" # Строго чистая версия

dfs = {name: pd.read_csv(os.path.join(base_path, f)).sort_values("index").reset_index(drop=True) for name, f in core_files.items()}
dfs["flaml_all27"] = pd.read_csv(os.path.join(base_path, flaml_path)).sort_values("index").reset_index(drop=True)
dfs["mlp_q10"] = pd.read_csv(os.path.join(base_path, mlp_path)).sort_values("index").reset_index(drop=True)

ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

# Задаем фиксированную пропорцию весов внутри ядра (по вашему лучшему сабмиту)
core_ratios = {
    "lgb_eng": 0.20, "lgb_dart": 0.20,
    "logreg": 0.21, "log_stacker_v3": 0.21,
    "tabm": 0.09, "flaml_v3": 0.09
}
core_ratio_sum = sum(core_ratios.values())

# Проверяем две лучшие степени p
p_options = [4.13, 4.24]
# Линейка весов для MLP (8 шагов от 1.5% до 6.7%)
mlp_weights = [0.015 + (j * 0.0074) for j in range(8)] 

sub_idx = 1
print(f"Запуск финальной сетки на {len(p_options) * len(mlp_weights)} сабмитов...")

for p in p_options:
    for w_mlp in mlp_weights:
        # Фиксируем лучший вес для FLAML (около 10.4%)
        w_flaml = 0.104
        
        # Остаток распределяем строго пропорционально внутри ядра
        core_leftover = 1.0 - w_flaml - w_mlp
        w = {name: (val / core_ratio_sum) * core_leftover for name, val in core_ratios.items()}
        
        # Считаем финальный бленд
        final_rank = (
            w["lgb_eng"] * (ranks["lgb_eng"]**p) +
            w["lgb_dart"] * (ranks["lgb_dart"]**p) +
            w["logreg"] * (ranks["logreg"]**p) +
            w["log_stacker_v3"] * (ranks["log_stacker_v3"]**p) +
            w["tabm"] * (ranks["tabm"]**p) +
            w["flaml_v3"] * (ranks["flaml_v3"]**p) +
            w_flaml * (ranks["flaml_all27"]**p) +
            w_mlp * (ranks["mlp_q10"]**p)
        )
        
        sub = base_df.copy()
        sub["score"] = rankdata(final_rank) / len(final_rank)
        
        sub_name = f"final_grid_{sub_idx:02d}__p_{p}__flaml_0.104__mlp_{w_mlp:.3f}.csv"
        sub.to_csv(os.path.join(output_path, sub_name), index=False)
        print(f"Создан [{sub_idx:02d}/16]: p={p}, вес MLP={w_mlp:.3f}")
        sub_idx += 1

print(f"\nВсе файлы сохранены в: {output_path}")


Запуск финальной сетки на 16 сабмитов...
Создан [01/16]: p=4.13, вес MLP=0.015
Создан [02/16]: p=4.13, вес MLP=0.022
Создан [03/16]: p=4.13, вес MLP=0.030
Создан [04/16]: p=4.13, вес MLP=0.037
Создан [05/16]: p=4.13, вес MLP=0.045
Создан [06/16]: p=4.13, вес MLP=0.052
Создан [07/16]: p=4.13, вес MLP=0.059
Создан [08/16]: p=4.13, вес MLP=0.067
Создан [09/16]: p=4.24, вес MLP=0.015
Создан [10/16]: p=4.24, вес MLP=0.022
Создан [11/16]: p=4.24, вес MLP=0.030
Создан [12/16]: p=4.24, вес MLP=0.037
Создан [13/16]: p=4.24, вес MLP=0.045
Создан [14/16]: p=4.24, вес MLP=0.052
Создан [15/16]: p=4.24, вес MLP=0.059
Создан [16/16]: p=4.24, вес MLP=0.067

Все файлы сохранены в: /home/jupyter/project/blends/final_all_in_attack


In [18]:
base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/final_p_tuning"
os.makedirs(output_path, exist_ok=True)

# 1. Ваше проверенное базовое ядро (6 моделей)
core_files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

flaml_path = "submissions_leak/flaml_all_train_fields_0.66582.csv"

dfs = {name: pd.read_csv(os.path.join(base_path, f)).sort_values("index").reset_index(drop=True) for name, f in core_files.items()}
dfs["flaml_all27"] = pd.read_csv(os.path.join(base_path, flaml_path)).sort_values("index").reset_index(drop=True)

ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

# Жесткие исторические пропорции ядра
core_ratios = {
    "lgb_eng": 0.20, "lgb_dart": 0.20,
    "logreg": 0.21, "log_stacker_v3": 0.21,
    "tabm": 0.09, "flaml_v3": 0.09
}
core_ratio_sum = sum(core_ratios.values())

# Фиксируем идеальный вес для FLAML
w_flaml = 0.100
core_leftover = 1.0 - w_flaml
w = {name: (val / core_ratio_sum) * core_leftover for name, val in core_ratios.items()}

# Сетка степеней p (7 шагов)
p_steps = [4.0, 4.1, 4.2, 4.3, 4.4, 4.5, 4.6]

print(f"Запуск тюнинга степени на {len(p_steps)} сабмитов...")

for idx, p in enumerate(p_steps, 1):
    final_rank = (
        w["lgb_eng"] * (ranks["lgb_eng"]**p) +
        w["lgb_dart"] * (ranks["lgb_dart"]**p) +
        w["logreg"] * (ranks["logreg"]**p) +
        w["log_stacker_v3"] * (ranks["log_stacker_v3"]**p) +
        w["tabm"] * (ranks["tabm"]**p) +
        w["flaml_v3"] * (ranks["flaml_v3"]**p) +
        w_flaml * (ranks["flaml_all27"]**p)
    )
    
    sub = base_df.copy()
    sub["score"] = rankdata(final_rank) / len(final_rank)
    
    sub_name = f"tuning_p_{idx:02d}__p_{p:.2f}__flaml_0.100.csv"
    sub.to_csv(os.path.join(output_path, sub_name), index=False)
    print(f"Создан сабмит [{idx}/7]: p={p:.2f}")

print(f"\nВсе файлы сохранены в: {output_path}")


Запуск тюнинга степени на 7 сабмитов...
Создан сабмит [1/7]: p=4.00
Создан сабмит [2/7]: p=4.10
Создан сабмит [3/7]: p=4.20
Создан сабмит [4/7]: p=4.30
Создан сабмит [5/7]: p=4.40
Создан сабмит [6/7]: p=4.50
Создан сабмит [7/7]: p=4.60

Все файлы сохранены в: /home/jupyter/project/blends/final_p_tuning


In [19]:
base_path = "/home/jupyter/project"
output_path = "/home/jupyter/project/blends/final_overdrive"
os.makedirs(output_path, exist_ok=True)

# Базовое ядро
core_files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}
flaml_path = "submissions_leak/flaml_all_train_fields_0.66582.csv"

dfs = {name: pd.read_csv(os.path.join(base_path, f)).sort_values("index").reset_index(drop=True) for name, f in core_files.items()}
dfs["flaml_all27"] = pd.read_csv(os.path.join(base_path, flaml_path)).sort_values("index").reset_index(drop=True)

ranks = {name: rankdata(df["score"]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][["index"]].copy()

# Исторические веса
core_ratios = {
    "lgb_eng": 0.20, "lgb_dart": 0.20, "logreg": 0.21, "log_stacker_v3": 0.21, "tabm": 0.09, "flaml_v3": 0.09
}
core_ratio_sum = sum(core_ratios.values())
w_flaml = 0.100
core_leftover = 1.0 - w_flaml
w = {name: (val / core_ratio_sum) * core_leftover for name, val in core_ratios.items()}

# Пробиваем потолок дальше
p_overdrive = [4.70, 4.80, 4.90]

print("Генерация финальных трех сабмитов...")
for idx, p in enumerate(p_overdrive, 1):
    final_rank = (
        w["lgb_eng"] * (ranks["lgb_eng"]**p) +
        w["lgb_dart"] * (ranks["lgb_dart"]**p) +
        w["logreg"] * (ranks["logreg"]**p) +
        w["log_stacker_v3"] * (ranks["log_stacker_v3"]**p) +
        w["tabm"] * (ranks["tabm"]**p) +
        w["flaml_v3"] * (ranks["flaml_v3"]**p) +
        w_flaml * (ranks["flaml_all27"]**p)
    )
    
    sub = base_df.copy()
    sub["score"] = rankdata(final_rank) / len(final_rank)
    
    sub_name = f"overdrive_p_{p:.2f}__flaml_0.100.csv"
    sub.to_csv(os.path.join(output_path, sub_name), index=False)
    print(f"Создан [{idx}/3]: p={p:.2f}")


Генерация финальных трех сабмитов...
Создан [1/3]: p=4.70
Создан [2/3]: p=4.80
Создан [3/3]: p=4.90
